
# Planet Imagery Square Tiler — COG Output + CSV Manifest

This notebook tiles irregular Planet imagery (e.g., circular/irregular AOIs) into **equal-size square chips** while preserving all bands. It is designed for **glacial lakes detection** (segmentation / object detection), but is general enough for other tasks.

### What it does

- Reads input GeoTIFF (e.g., PlanetScope **analytic_8b_sr**).
- Generates fixed-size square tiles with **boundless** reads and padding using **nodata** so tiles at imagery borders remain consistent in size.
- Optionally reads **UDM2** to consider only **'clear'** pixels when computing valid coverage.
- **Skips** tiles whose valid coverage is below a chosen threshold (to avoid mostly empty/padded chips).
- Writes each tile as a **Cloud-Optimized GeoTIFF (COG)** when the GDAL COG driver is available; otherwise writes a well-tiled GeoTIFF with overviews (safe fallback).
- Produces a **GeoJSON index** (one feature per tile footprint) and a **CSV manifest** with useful attributes.
- Enforces **8-band** input (optional, recommended for PlanetScope 8-band SR).

### Why these choices?

- **Windowed reads** + **boundless** padding are the standard, memory-safe way to chip large rasters.
- **Coverage threshold** filters out mostly padded tiles; you control the signal/noise trade-off.
- **Overlap (stride < tile)** keeps spatial context at tile edges—useful for segmentation/detection.
- **UDM2** masks: counting only “clear” pixels (no cloud/haze/shadow/snow) lifts data quality.
- **COGs** make downstream I/O (local or cloud) faster and more interoperable.



## 0) Requirements & configuration

This notebook uses:
- `rasterio`, `numpy`, `geopandas`, `shapely`, `pandas`
- GDAL ≥ 3.1 enables the **COG** driver (preferred). If that's not available, the notebook falls back to a standard tiled GeoTIFF + internal overviews.

> **Tip:** If your Planet product is `analytic_8b_sr_udm2`, you will have a matching UDM2 file (e.g., `*_udm2_*.tif`). Pass its path below to score tiles by **clear** pixels.


In [1]:

# --- User config ---
IN_RASTER_DIR = r"D:\Thesis\glacial_lake_detection_thesis\data\raw\planet\train\planet_downloads_PSScene_rgb_2020\orders_all"  # <-- change
UDM2_RASTER = None            # or None
OUT_DIR = r"D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256"                                                                      # directory to create

TILE_SIZE_PX = 256           # tile width = height in pixels
STRIDE_PX = 205              # < TILE_SIZE_PX -> adds overlap (~20%)
COVERAGE_THRESHOLD = 0.60    # keep tiles with >= 60% valid pixels
REQUIRE_8_BANDS = False       # set False to allow non-8-band inputs
OVERWRITE = False            # set True to re-generate if files exist

# COG/GTiff writing preferences
COG_COMPRESS = "DEFLATE"     # for COGs
GTIFF_COMPRESS = "DEFLATE"   # for fallback GTiff
BIGTIFF = "IF_SAFER"         # safe BigTIFF policy

# UDM2 band name to search for; if not found, band 1 is used as fallback
UDM2_CLEAR_BAND_NAME = "clear"
UDM2_CLEAR_BAND_INDEX_HINT = None  # e.g., 0 for the first band (0-based) if you know it


In [2]:

from __future__ import annotations
from pathlib import Path
from typing import Optional, Tuple, List
import warnings
import json

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, mapping

import rasterio
from rasterio.windows import Window
from rasterio.transform import Affine
from rasterio.enums import Resampling
from rasterio.errors import NotGeoreferencedWarning
from rasterio.io import DatasetReader
from rasterio.shutil import copy as rio_copy

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)
print("rasterio", rasterio.__version__)


rasterio 1.4.3



## 1) Helper functions

- Window generation and boundless reads
- Valid mask from `nodata`
- Optional UDM2 “clear” mask
- Tile polygon footprint
- COG writer with safe fallback


In [3]:

def _tile_windows(width: int, height: int, tile_px: int, stride_px: Optional[int] = None) -> List[Window]:
    if stride_px is None:
        stride_px = tile_px
    col_starts = list(range(0, width, stride_px))
    row_starts = list(range(0, height, stride_px))
    return [Window(c, r, tile_px, tile_px) for r in row_starts for c in col_starts]


def _read_tile(ds: DatasetReader, w: Window) -> Tuple[np.ndarray, Affine]:
    tile = ds.read(
        window=w,
        boundless=True,
        fill_value=ds.nodata if ds.nodata is not None else 0
    )
    transform = rasterio.windows.transform(w, ds.transform)
    return tile, transform


def _valid_mask_from_dataset(ds: DatasetReader, tile: np.ndarray) -> np.ndarray:
    nd = ds.nodata
    if nd is not None:
        invalid = np.all(tile == nd, axis=0)
    else:
        #if np.issubdtype(tile.dtype, np.floating):
            #invalid = np.all(np.isnan(tile), axis=0)
        #else:
            #invalid = np.zeros(tile.shape[1:], dtype=bool)
            invalid = np.all(tile == 0, axis=0)
    return ~invalid


def _read_udm2_clear(udm2_ds: DatasetReader, w: Window, clear_band_index: int) -> np.ndarray:
    arr = udm2_ds.read(
        indexes=clear_band_index + 1,
        window=w,
        boundless=True,
        fill_value=0
    )
    return arr[np.newaxis, ...]


def _valid_mask_with_udm2(base_valid: np.ndarray,
                          udm2_tile: Optional[np.ndarray],
                          clear_band_index: Optional[int]) -> np.ndarray:
    if udm2_tile is None or clear_band_index is None:
        return base_valid
    clear = udm2_tile[0].astype(bool)  # since _read_udm2_clear returns shape (1,H,W)
    return base_valid & clear


def _tile_polygon(transform: Affine, tile_px: int) -> Polygon:
    x0, y0 = transform * (0, 0)
    x1, y1 = transform * (tile_px, tile_px)
    return Polygon([(x0, y0), (x1, y0), (x1, y1), (x0, y1)])


def _gdal_has_cog_driver() -> bool:
    try:
        from rasterio.env import GDALVersion
        from rasterio._env import get_gdal_config
    except Exception:
        pass
    try:
        import rasterio
        drivers = rasterio.env.Env().drivers()
        # Driver presence check
        return "COG" in drivers
    except Exception:
        return False


def _write_cog_or_tiff(src_array: np.ndarray,
                       out_path: Path,
                       transform: Affine,
                       crs,
                       dtype,
                       compress_cog="DEFLATE",
                       compress_gtiff="DEFLATE",
                       bigtiff="IF_SAFER",
                       nodata=None,
                       colorinterp=None):
    """Write as COG if available, else as tiled GTiff + overviews."""
    bands, height, width = src_array.shape

    # Base profile
    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": bands,
        "dtype": dtype,
        "crs": crs,
        "transform": transform,
        "tiled": True,
        "blockxsize": min(512, width),
        "blockysize": min(512, height),
        "compress": compress_gtiff,
        "BIGTIFF": bigtiff,
    }
    if nodata is not None:
        profile["nodata"] = nodata

    # First write to a temporary GTiff
    tmp_path = out_path.with_suffix(".tmp.tif")
    with rasterio.open(tmp_path, "w", **profile) as dst:
        dst.write(src_array)
        if colorinterp:
            try:
                dst.colorinterp = colorinterp
            except Exception:
                pass
        # Build internal overviews for better COG fallback performance
        factors = []
        level = 2
        while max(height // level, width // level) >= 512:
            factors.append(level)
            level *= 2
        if factors:
            dst.build_overviews(factors, Resampling.nearest)
            dst.update_tags(ns="rio_overview", resampling="nearest")

    # Try COG conversion via GDAL driver
    if _gdal_has_cog_driver():
        rio_copy(
            tmp_path.as_posix(),
            out_path.as_posix(),
            driver="COG",
            compress=compress_cog,
            BIGTIFF=bigtiff,
            NUM_THREADS="ALL_CPUS"
        )
        Path(tmp_path).unlink(missing_ok=True)
    else:
        # Fallback: keep the overviews-enabled GTiff and rename
        tmp_path.rename(out_path)



## 2) Main tiler

- Preserves **all bands** by default; optionally enforces **8-band** input.
- Computes valid coverage using nodata and (optionally) UDM2 **clear**.
- Writes each kept tile as **COG** (or fallback GTiff), and records entries for GeoJSON + CSV.


In [4]:

def tile_planet_raster_to_cogs(
    in_raster: str | Path,
    out_dir: str | Path,
    tile_size_px: int = 512,
    stride_px: Optional[int] = None,
    coverage_threshold: float = 0.6,
    udm2_path: Optional[str | Path] = None,
    udm2_clear_band_name: str = "clear",
    clear_band_index_hint: Optional[int] = None,
    require_8_bands: bool = True,
    overwrite: bool = False,
    cog_compress: str = "DEFLATE",
    gtiff_compress: str = "DEFLATE",
    bigtiff: str = "IF_SAFER",
):
    in_raster = Path(in_raster)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Open source
    with rasterio.open(in_raster) as ds:
        if require_8_bands and ds.count != 8:
            raise RuntimeError(f"Input has {ds.count} bands, but 8 were required. Set require_8_bands=False to allow." )
        width, height = ds.width, ds.height
        crs = ds.crs
        dtype = ds.dtypes[0]
        nodata = ds.nodata
        colorinterp = None
        try:
            colorinterp = ds.colorinterp
        except Exception:
            pass

        # UDM2 setup
        udm2_ds = None
        clear_band_index = None
        if udm2_path is not None:
            udm2_ds = rasterio.open(udm2_path)
            if clear_band_index_hint is not None:
                clear_band_index = clear_band_index_hint
            else:
                # Attempt to find 'clear' band by description/name
                clear_band_index = None
                descs = [udm2_ds.descriptions[i] if udm2_ds.descriptions else None for i in range(udm2_ds.count)]
                for i, d in enumerate(descs):
                    if d and (udm2_clear_band_name.lower() in d.lower()):
                        clear_band_index = i
                        break
                if clear_band_index is None:
                    clear_band_index = 0  # fallback

        windows = _tile_windows(width, height, tile_size_px, stride_px)
        features = []
        manifest_rows = []

        for idx, w in enumerate(windows):
            tile, tform = _read_tile(ds, w)
            base_valid = _valid_mask_from_dataset(ds, tile)

            udm2_tile = None
            if udm2_ds is not None:
                udm2_tile = _read_udm2_clear(udm2_ds, w, clear_band_index)

            valid_mask = _valid_mask_with_udm2(base_valid, udm2_tile, 0 if udm2_tile is not None else None)
            valid_fraction = float(valid_mask.sum()) / valid_mask.size

            if valid_fraction < coverage_threshold:
                continue

            out_name = f"{in_raster.stem}_r{int(w.row_off)}_c{int(w.col_off)}_{tile_size_px}px.tif"
            out_path = out_dir / out_name

            if (not out_path.exists()) or overwrite:
                _write_cog_or_tiff(
                    src_array=tile,
                    out_path=out_path,
                    transform=tform,
                    crs=crs,
                    dtype=dtype,
                    compress_cog=cog_compress,
                    compress_gtiff=gtiff_compress,
                    bigtiff=bigtiff,
                    nodata=nodata,
                    colorinterp=colorinterp
                )

            # Geo feature & manifest
            poly = _tile_polygon(tform, tile_size_px)
            features.append({
                "tile_path": str(out_path),
                "row_off": int(w.row_off),
                "col_off": int(w.col_off),
                "valid_fraction": valid_fraction,
                "geometry": poly
            })
            # CSV manifest row
            bounds = poly.bounds
            manifest_rows.append({
                "tile_path": str(out_path),
                "row_off": int(w.row_off),
                "col_off": int(w.col_off),
                "valid_fraction": valid_fraction,
                "minx": bounds[0], "miny": bounds[1], "maxx": bounds[2], "maxy": bounds[3],
                "tile_size_px": tile_size_px,
                "stride_px": stride_px if stride_px is not None else tile_size_px,
                "bands": ds.count,
                "dtype": dtype,
                "crs": str(crs)
            })

        if udm2_ds is not None:
            udm2_ds.close()

    # Write index + CSV
    gdf = gpd.GeoDataFrame(features, geometry="geometry", crs=crs)
    idx_path = Path(out_dir) / f"{in_raster.stem}_tiles_index.geojson"
    gdf.to_file(idx_path, driver="GeoJSON")

    manifest_df = pd.DataFrame(manifest_rows)
    csv_path = Path(out_dir) / f"{in_raster.stem}_tiles_manifest.csv"
    manifest_df.to_csv(csv_path, index=False)

    return gdf, manifest_df, idx_path, csv_path



## 3) Run tiling

Adjust the **User config** cell, then execute the cell below.


In [6]:
# specify folder path
input_folder = Path(IN_RASTER_DIR)

# get all files (any type) and make a list of their full paths
input_file_paths = [str(p.resolve()) for p in input_folder.glob("*") if p.is_file()]

for input_file_path in input_file_paths:

    gdf, manifest_df, idx_path, csv_path = tile_planet_raster_to_cogs(
        in_raster=input_file_path,
        out_dir=OUT_DIR, #+ f"/{Path(input_file_path).stem}",
        tile_size_px=TILE_SIZE_PX,
        stride_px=STRIDE_PX,
        coverage_threshold=COVERAGE_THRESHOLD,
        udm2_path=UDM2_RASTER if (UDM2_RASTER and Path(UDM2_RASTER).exists()) else None,
        udm2_clear_band_name=UDM2_CLEAR_BAND_NAME,
        clear_band_index_hint=UDM2_CLEAR_BAND_INDEX_HINT,
        require_8_bands=REQUIRE_8_BANDS,
        overwrite=OVERWRITE,
        cog_compress=COG_COMPRESS,
        gtiff_compress=GTIFF_COMPRESS,
        bigtiff=BIGTIFF,
    )

    print(f"Wrote index: {idx_path}")
    print(f"Wrote CSV  : {csv_path}")
    display(gdf.head())
    display(manifest_df.head())


Wrote index: D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256\20200901_045513_72_2277_3B_Visual_clip_reproject_file_format_tiles_index.geojson
Wrote CSV  : D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256\20200901_045513_72_2277_3B_Visual_clip_reproject_file_format_tiles_manifest.csv


,tile_path,row_off,col_off,valid_fraction,geometry
0,D:\Thesis\glacial_lake_detection_thesis\data\p...,0,615,0.829742,"POLYGON ((75.87401 34.90523, 75.88154 34.90523..."
1,D:\Thesis\glacial_lake_detection_thesis\data\p...,0,820,0.782730,"POLYGON ((75.88004 34.90523, 75.88756 34.90523..."
2,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,410,0.835922,"POLYGON ((75.86799 34.8992, 75.87551 34.8992, ..."
3,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,615,1.000000,"POLYGON ((75.87401 34.8992, 75.88154 34.8992, ..."
4,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,820,1.000000,"POLYGON ((75.88004 34.8992, 75.88756 34.8992, ..."


,tile_path,row_off,col_off,valid_fraction,minx,miny,maxx,maxy,tile_size_px,stride_px,bands,dtype,crs
0,D:\Thesis\glacial_lake_detection_thesis\data\p...,0,615,0.829742,75.874012,34.897702,75.881537,34.905227,256,205,4,uint8,EPSG:4326
1,D:\Thesis\glacial_lake_detection_thesis\data\p...,0,820,0.782730,75.880038,34.897702,75.887564,34.905227,256,205,4,uint8,EPSG:4326
2,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,410,0.835922,75.867985,34.891675,75.875511,34.899201,256,205,4,uint8,EPSG:4326
3,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,615,1.000000,75.874012,34.891675,75.881537,34.899201,256,205,4,uint8,EPSG:4326
4,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,820,1.000000,75.880038,34.891675,75.887564,34.899201,256,205,4,uint8,EPSG:4326


Wrote index: D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256\20200908_050721_26_2259_3B_Visual_clip_reproject_file_format_tiles_index.geojson
Wrote CSV  : D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256\20200908_050721_26_2259_3B_Visual_clip_reproject_file_format_tiles_manifest.csv


,tile_path,row_off,col_off,valid_fraction,geometry
0,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,410,0.854401,"POLYGON ((73.37262 35.39405, 73.38077 35.39405..."
1,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,615,0.712006,"POLYGON ((73.37914 35.39405, 73.38729 35.39405..."
2,D:\Thesis\glacial_lake_detection_thesis\data\p...,410,205,0.630829,"POLYGON ((73.3661 35.38753, 73.37425 35.38753,..."
3,D:\Thesis\glacial_lake_detection_thesis\data\p...,410,410,1.000000,"POLYGON ((73.37262 35.38753, 73.38077 35.38753..."
4,D:\Thesis\glacial_lake_detection_thesis\data\p...,410,615,1.000000,"POLYGON ((73.37914 35.38753, 73.38729 35.38753..."


...

,tile_path,row_off,col_off,valid_fraction,geometry
0,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,1640,0.950089,"POLYGON ((74.09334 35.85545, 74.10143 35.85545..."
1,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,1845,1.000000,"POLYGON ((74.09982 35.85545, 74.10791 35.85545..."
2,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,2050,0.706009,"POLYGON ((74.1063 35.85545, 74.1144 35.85545, ..."
3,D:\Thesis\glacial_lake_detection_thesis\data\p...,410,1230,0.818527,"POLYGON ((74.08037 35.84897, 74.08847 35.84897..."
4,D:\Thesis\glacial_lake_detection_thesis\data\p...,410,1435,1.000000,"POLYGON ((74.08686 35.84897, 74.09495 35.84897..."


,tile_path,row_off,col_off,valid_fraction,minx,miny,maxx,maxy,tile_size_px,stride_px,bands,dtype,crs
0,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,1640,0.950089,74.093338,35.847354,74.101432,35.855449,256,205,4,uint8,EPSG:4326
1,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,1845,1.000000,74.099820,35.847354,74.107915,35.855449,256,205,4,uint8,EPSG:4326
2,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,2050,0.706009,74.106302,35.847354,74.114397,35.855449,256,205,4,uint8,EPSG:4326
3,D:\Thesis\glacial_lake_detection_thesis\data\p...,410,1230,0.818527,74.080373,35.840872,74.088468,35.848966,256,205,4,uint8,EPSG:4326
4,D:\Thesis\glacial_lake_detection_thesis\data\p...,410,1435,1.000000,74.086855,35.840872,74.094950,35.848966,256,205,4,uint8,EPSG:4326


Wrote index: D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256\20200925_050740_48_222f_3B_Visual_clip_reproject_file_format_tiles_index.geojson
Wrote CSV  : D:\Thesis\glacial_lake_detection_thesis\data\processed\train\rgb_outputs\chips_out_256\20200925_050740_48_222f_3B_Visual_clip_reproject_file_format_tiles_manifest.csv


,tile_path,row_off,col_off,valid_fraction,geometry
0,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,1640,0.885468,"POLYGON ((73.23742 36.08263, 73.24523 36.08263..."
1,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,1845,1.000000,"POLYGON ((73.24368 36.08263, 73.25149 36.08263..."
2,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,2050,0.996796,"POLYGON ((73.24993 36.08263, 73.25774 36.08263..."
3,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,2255,0.927032,"POLYGON ((73.25619 36.08263, 73.264 36.08263, ..."
4,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,2460,0.809891,"POLYGON ((73.26244 36.08263, 73.27025 36.08263..."


,tile_path,row_off,col_off,valid_fraction,minx,miny,maxx,maxy,tile_size_px,stride_px,bands,dtype,crs
0,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,1640,0.885468,73.237422,36.074819,73.245233,36.082629,256,205,4,uint8,EPSG:4326
1,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,1845,1.000000,73.243677,36.074819,73.251487,36.082629,256,205,4,uint8,EPSG:4326
2,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,2050,0.996796,73.249931,36.074819,73.257741,36.082629,256,205,4,uint8,EPSG:4326
3,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,2255,0.927032,73.256185,36.074819,73.263996,36.082629,256,205,4,uint8,EPSG:4326
4,D:\Thesis\glacial_lake_detection_thesis\data\p...,205,2460,0.809891,73.262440,36.074819,73.270250,36.082629,256,205,4,uint8,EPSG:4326



## 4) Notes & tuning tips

- **8 bands guaranteed?** This notebook, by default, requires **8 bands**; if you pass a non-8-band input, it will raise an error. To relax this, set `REQUIRE_8_BANDS=False` — the code still preserves **all** available bands.
- **Stride/overlap:** For segmentation, try 10–25% overlap (`STRIDE_PX ≈ 0.75–0.9 * TILE_SIZE_PX`). More overlap = more context, more chips.
- **Coverage threshold:** Start with `0.6`; raise if your AOIs are very irregular and you want fewer partially-padded tiles.
- **COG vs GTiff:** If GDAL COG driver is available, outputs are true COGs. Otherwise, you still get tiled GTiffs with internal overviews (fast and compatible).
- **Performance:** Tile IO is the bottleneck. Consider parallelizing across **multiple input rasters** using joblib / multiprocessing if needed.
